# ThessLink RL — Training Notebook

Εκπαίδευση και αξιολόγηση των αλγορίθμων **Q-Learning**, **DQN**, **PPO** σε grids 8×8, 32×32, 64×64.

| Grid | Q-Learning | DQN | PPO |
|------|-----------|-----|-----|
| 8×8  | `nav_qtable_8.pkl` | `nav_dqn_8.zip` | `nav_ppo_8.zip` |
| 32×32 | `nav_qtable_32.pkl` | `nav_dqn_32.zip` | `nav_ppo_32.zip` |
| 64×64 | `nav_qtable_64.pkl` | `nav_dqn_64.zip` | `nav_ppo_64.zip` |

In [ ]:
import sys
from pathlib import Path

# Make sure the project root and lb-foraging are on the path
ROOT = Path(".").resolve()
LBF = ROOT / "lb-foraging"
for p in [str(ROOT), str(LBF)]:
    if p not in sys.path:
        sys.path.insert(0, p)

print("Project root:", ROOT)

In [ ]:
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for notebooks
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from IPython.display import display, Image

from navigation_train import (
    train_ppo, train_dqn, train_qlearning,
    _eval_navigation, _discretize_nav, _NAV_ACTIONS,
    MODEL_DIR, PLOT_DIR,
)
from poi_environment import PoINavigationEnv

print("Imports OK")

---
## Βοηθητικές συναρτήσεις

In [ ]:
import pickle

GRID_SIZES = [8, 32, 64]
ALGOS = ["ppo", "dqn", "qlearning"]


def model_exists(algo: str, grid_size: int) -> bool:
    """Check if a trained model file exists."""
    tag = str(grid_size)
    if algo == "ppo":
        return (MODEL_DIR / "ppo" / f"nav_ppo_{tag}.zip").exists()
    elif algo == "dqn":
        return (MODEL_DIR / "dqn" / f"nav_dqn_{tag}.zip").exists()
    else:
        return (MODEL_DIR / "qlearning" / f"nav_qtable_{tag}.pkl").exists()


def plot_exists(algo: str, grid_size: int) -> bool:
    return (PLOT_DIR / f"training_plot_nav_{algo}_{grid_size}.png").exists()


def show_status():
    """Print a table showing which models and plots exist."""
    header = f"{'':12}" + "".join(f"{g:>8}" for g in GRID_SIZES)
    print(header)
    print("-" * (12 + 8 * len(GRID_SIZES)))
    for algo in ALGOS:
        row = f"{algo:<12}"
        for g in GRID_SIZES:
            m = "M" if model_exists(algo, g) else "."
            p = "P" if plot_exists(algo, g) else "."
            row += f"  [{m}{p}]  "
        print(row)
    print("\nM=model exists  P=plot exists  .=missing")


show_status()

---
## Training

Επίλεξε αλγόριθμο και grid size και τρέξε το αντίστοιχο cell.

> **Steps/episodes για ~5 λεπτά training ανά grid (εκτίμηση):**
> | Grid | PPO steps | DQN steps | Q-Learning episodes |
> |------|-----------|-----------|--------------------|
> | 8×8  | 30 000    | 30 000    | 150 000            |
> | 32×32 | 80 000   | 80 000    | 300 000            |
> | 64×64 | 150 000  | 150 000   | 500 000            |
>
> Οι τιμές αυτές είναι **προεπιλεγμένες** στα training cells παρακάτω.

### PPO

In [ ]:
# ~5 min per grid (SB3 MLP ~8-12k steps/sec minus eval overhead)
_PPO_STEPS = {8: 30_000, 32: 80_000, 64: 150_000}

GRID = 8   # ← αλλαγή: 8, 32, ή 64
STEPS = _PPO_STEPS[GRID]

plot_path = PLOT_DIR / f"training_plot_nav_ppo_{GRID}.png"
train_ppo(
    total_timesteps=STEPS,
    seed=42,
    eval_freq=max(STEPS // 20, 1000),
    plot_path=plot_path,
    grid_size=(GRID, GRID),
)
show_status()

### DQN

In [ ]:
# ~5 min per grid (SB3 MLP ~8-12k steps/sec minus eval overhead)
_DQN_STEPS = {8: 30_000, 32: 80_000, 64: 150_000}

GRID = 8   # ← αλλαγή: 8, 32, ή 64
STEPS = _DQN_STEPS[GRID]

plot_path = PLOT_DIR / f"training_plot_nav_dqn_{GRID}.png"
train_dqn(
    total_timesteps=STEPS,
    seed=42,
    eval_freq=max(STEPS // 20, 1000),
    plot_path=plot_path,
    grid_size=(GRID, GRID),
)
show_status()

### Q-Learning

In [ ]:
# ~5 min per grid (Q-Learning ~50-150k episodes/sec, no neural net)
_QL_EPISODES = {8: 150_000, 32: 300_000, 64: 500_000}

GRID = 8   # ← αλλαγή: 8, 32, ή 64
EPISODES = _QL_EPISODES[GRID]

plot_path = PLOT_DIR / f"training_plot_nav_qlearning_{GRID}.png"
train_qlearning(
    total_episodes=EPISODES,
    seed=42,
    eval_freq=max(EPISODES // 20, 5000),
    plot_path=plot_path,
    grid_size=(GRID, GRID),
)
show_status()

---
## Εμφάνιση Plots

### Ένα συγκεκριμένο plot

In [ ]:
ALGO = "ppo"   # ← ppo, dqn, qlearning
GRID = 8       # ← 8, 32, 64

path = PLOT_DIR / f"training_plot_nav_{ALGO}_{GRID}.png"
if path.exists():
    img = mpimg.imread(str(path))
    fig, ax = plt.subplots(figsize=(9, 11))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print(f"Plot not found: {path}")

### Όλα τα plots — grid 3×3 (αλγόριθμος × grid size)

In [ ]:
fig, axes = plt.subplots(
    len(ALGOS), len(GRID_SIZES),
    figsize=(7 * len(GRID_SIZES), 11 * len(ALGOS)),
)

for row, algo in enumerate(ALGOS):
    for col, grid in enumerate(GRID_SIZES):
        ax = axes[row][col]
        path = PLOT_DIR / f"training_plot_nav_{algo}_{grid}.png"
        if path.exists():
            img = mpimg.imread(str(path))
            ax.imshow(img)
            ax.set_title(f"{algo.upper()} — {grid}×{grid}", fontsize=14, fontweight="bold")
        else:
            ax.text(0.5, 0.5, f"Not trained yet\n({algo} {grid}×{grid})",
                    ha="center", va="center", fontsize=12, color="gray",
                    transform=ax.transAxes)
            ax.set_facecolor("#f5f5f5")
            ax.set_title(f"{algo.upper()} — {grid}×{grid}", fontsize=14, color="lightgray")
        ax.axis("off")

plt.suptitle("ThessLink RL — Learning Curves (3×3)", fontsize=18, fontweight="bold", y=1.002)
plt.tight_layout()
plt.show()

---
## Αξιολόγηση (Evaluation)

### Ένα μοντέλο

In [ ]:
ALGO = "ppo"   # ← ppo, dqn, qlearning
GRID = 8       # ← 8, 32, 64
N_EPISODES = 200

tag = str(GRID)
grid_size = (GRID, GRID)

if not model_exists(ALGO, GRID):
    print(f"Model not found: {ALGO} {GRID}×{GRID}. Train it first.")
else:
    if ALGO == "ppo":
        from stable_baselines3 import PPO
        m = PPO.load(str(MODEL_DIR / "ppo" / f"nav_ppo_{tag}"))
        predict = lambda obs: int(m.predict(obs, deterministic=True)[0])
    elif ALGO == "dqn":
        from stable_baselines3 import DQN
        m = DQN.load(str(MODEL_DIR / "dqn" / f"nav_dqn_{tag}"))
        predict = lambda obs: int(m.predict(obs, deterministic=True)[0])
    else:
        with open(MODEL_DIR / "qlearning" / f"nav_qtable_{tag}.pkl", "rb") as f:
            q_table = pickle.load(f)
        predict = lambda obs: int(np.argmax(q_table.get(_discretize_nav(obs), np.zeros(_NAV_ACTIONS))))

    stats = _eval_navigation(predict, n_episodes=N_EPISODES, grid_size=grid_size)
    print(f"\n{'='*40}")
    print(f"  {ALGO.upper()} — {GRID}×{GRID} ({N_EPISODES} episodes)")
    print(f"{'='*40}")
    print(f"  Success rate : {stats['success_rate']:.1%}")
    print(f"  Mean reward  : {stats['mean_reward']:.4f}")
    print(f"  Mean steps   : {stats['mean_steps']:.1f}")
    print(f"  Agreement    : {stats['agreement']:.1%}")

### Σύγκριση όλων των μοντέλων (πίνακας)

In [ ]:
N_EPISODES = 100  # ← μειώστε για ταχύτητα

results = []

for algo in ALGOS:
    for grid in GRID_SIZES:
        tag = str(grid)
        grid_size = (grid, grid)
        if not model_exists(algo, grid):
            results.append({"algo": algo, "grid": grid, "status": "not trained",
                            "success": None, "reward": None, "steps": None, "agreement": None})
            continue

        print(f"Evaluating {algo.upper()} {grid}×{grid}...", end=" ", flush=True)
        if algo == "ppo":
            from stable_baselines3 import PPO
            m = PPO.load(str(MODEL_DIR / "ppo" / f"nav_ppo_{tag}"))
            predict = lambda obs, _m=m: int(_m.predict(obs, deterministic=True)[0])
        elif algo == "dqn":
            from stable_baselines3 import DQN
            m = DQN.load(str(MODEL_DIR / "dqn" / f"nav_dqn_{tag}"))
            predict = lambda obs, _m=m: int(_m.predict(obs, deterministic=True)[0])
        else:
            with open(MODEL_DIR / "qlearning" / f"nav_qtable_{tag}.pkl", "rb") as f:
                qt = pickle.load(f)
            predict = lambda obs, _qt=qt: int(np.argmax(_qt.get(_discretize_nav(obs), np.zeros(_NAV_ACTIONS))))

        stats = _eval_navigation(predict, n_episodes=N_EPISODES, grid_size=grid_size)
        results.append({"algo": algo, "grid": grid, "status": "ok",
                        "success": stats["success_rate"], "reward": stats["mean_reward"],
                        "steps": stats["mean_steps"], "agreement": stats["agreement"]})
        print(f"success={stats['success_rate']:.1%}")

# Pretty table
print(f"\n{'Algo':<12} {'Grid':>6} {'Success':>9} {'Reward':>9} {'Steps':>8} {'Agreement':>11}")
print("-" * 60)
for r in results:
    if r["status"] == "not trained":
        print(f"{r['algo']:<12} {r['grid']:>4}×{r['grid']:<4}  {'(not trained)':>40}")
    else:
        print(f"{r['algo']:<12} {r['grid']:>4}×{r['grid']:<4}  "
              f"{r['success']:>8.1%}  {r['reward']:>9.4f}  {r['steps']:>7.1f}  {r['agreement']:>10.1%}")

### Σύγκριση success rate — bar chart

In [ ]:
trained = [r for r in results if r["status"] == "ok"]

if not trained:
    print("Δεν υπάρχουν trained models για σύγκριση.")
else:
    algo_colors = {"ppo": "tab:blue", "dqn": "tab:green", "qlearning": "tab:orange"}
    x = np.arange(len(GRID_SIZES))
    width = 0.25

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    for i, algo in enumerate(ALGOS):
        vals_success = []
        vals_reward = []
        for grid in GRID_SIZES:
            r = next((r for r in trained if r["algo"] == algo and r["grid"] == grid), None)
            vals_success.append(r["success"] if r else 0)
            vals_reward.append(r["reward"] if r else 0)

        offset = (i - 1) * width
        axes[0].bar(x + offset, vals_success, width, label=algo.upper(),
                    color=algo_colors[algo], alpha=0.85)
        axes[1].bar(x + offset, vals_reward, width, label=algo.upper(),
                    color=algo_colors[algo], alpha=0.85)

    axes[0].set_title("Success Rate", fontsize=13, fontweight="bold")
    axes[0].set_ylabel("Success Rate")
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f"{g}×{g}" for g in GRID_SIZES])
    axes[0].set_ylim(0, 1.1)
    axes[0].legend()
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].set_title("Mean Reward", fontsize=13, fontweight="bold")
    axes[1].set_ylabel("Mean Episode Reward")
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f"{g}×{g}" for g in GRID_SIZES])
    axes[1].legend()
    axes[1].grid(axis="y", alpha=0.3)

    plt.suptitle("ThessLink RL — Algorithm Comparison", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.show()